# Notebook 01 -- Raw Data Overview


> **Note:** This notebook is pre-run. The raw data lives on a local drive at and is not included in this repository. To reproduce, one has to extract the data from dsa database first and then pass the path to the raw data to dedicated code blocks.
>
> Row counts are taken from the processing log (`02_data_cleaning/logs/data_profile.md`) rather than re-scanning 120 GB. All other cells read from a single representative daily file.

---

This notebook shows the data **before any processing** -- as originally downloaded from the DSA Transparency Database. It demonstrates the full raw schema, what the data looks like globally, and how much information was narrowed down in subsequent steps.

## 1. What is DSA Statement of Reasons Data?

Under **Article 17 of the EU Digital Services Act (DSA)**, Very Large Online Platforms (VLOPs) must submit a **Statement of Reasons (SoR)** to the [DSA Transparency Database](https://transparency.dsa.ec.europa.eu/) each time they take a content moderation decision.



The raw data was downloaded using the `dsa-tdb` extraction tool, which produced **365 daily dump folders per platform** -- one for each day of 2025. Each folder contains chunked parquet files.

```
...\dsa-data\
    tiktok___full\daily_dumps_chunked\
        sor-tiktok-2025-01-01-full\
            part-0000.parquet
            part-0001.parquet
            ...
        sor-tiktok-2025-01-02-full\
            ...
    x___full\daily_dumps_chunked\
        sor-x-2025-01-01-full\
            ...
```

This thesis analyzes SoR data for **TikTok** and **X (formerly Twitter)** for the calendar year **2025**.

In [9]:
from pathlib import Path
import polars as pl

# Raw data on portable drive
TIKTOK_RAW_DIR = Path(r"E:\dsa-data\tiktok___full\daily_dumps_chunked")
X_RAW_DIR = Path(r"E:\dsa-data\x___full\daily_dumps_chunked")

# Single representative daily file -- used for schema and sample cells
TIKTOK_SAMPLE_FILE = TIKTOK_RAW_DIR / "sor-tiktok-2025-01-01-full" / "part-0000.parquet"
X_SAMPLE_FILE = X_RAW_DIR / "sor-x-2025-01-01-full" / "part-0000.parquet"

for label, path in [("TikTok raw dir", TIKTOK_RAW_DIR), ("X raw dir", X_RAW_DIR)]:
    print(f"{label}: {'OK' if path.exists() else 'NOT FOUND -- connect portable drive'}")

TikTok raw dir: OK
X raw dir: OK


## 2. Dataset Scale

Row counts are from the processing log (`02_data_cleaning/logs/data_profile.md`), recorded at extraction time.

| Platform | Daily folders | Total raw rows |
|---|---|---|
| TikTok | 365 | 1,002,729,268 |
| X | 365 | 670,093 |

The scale difference (~1,500:1 at raw level).

## 3. Raw Schema

The raw files contain **36 columns**. The thesis uses only 10 of these (selected during the filtering step -- see Notebook 02).

In [10]:
raw_schema = pl.read_parquet_schema(str(TIKTOK_SAMPLE_FILE))
print(f"Total columns in raw data: {len(raw_schema)}\n")
print(f"{'Column':<45} {'Type'}")
print("-" * 65)
for col, dtype in raw_schema.items():
    print(f"{col:<45} {dtype}")

Total columns in raw data: 37

Column                                        Type
-----------------------------------------------------------------
uuid                                          String
decision_visibility                           String
decision_visibility_other                     String
end_date_visibility_restriction               Datetime(time_unit='ms', time_zone=None)
decision_monetary                             String
decision_monetary_other                       String
end_date_monetary_restriction                 Datetime(time_unit='ms', time_zone=None)
decision_provision                            String
end_date_service_restriction                  Datetime(time_unit='ms', time_zone=None)
decision_account                              String
end_date_account_restriction                  Datetime(time_unit='ms', time_zone=None)
account_type                                  String
decision_ground                               String
decision_ground_reference_u

[DSA Official Schema Documentation](https://transparency.dsa.ec.europa.eu/page/api-documentation)

## 4. Sample Raw Rows

### TikTok (2025-01-01, first 5 rows)

In [6]:
pl.scan_parquet(str(TIKTOK_SAMPLE_FILE)).head(5).collect()

uuid,decision_visibility,decision_visibility_other,end_date_visibility_restriction,decision_monetary,decision_monetary_other,end_date_monetary_restriction,decision_provision,end_date_service_restriction,decision_account,end_date_account_restriction,account_type,decision_ground,decision_ground_reference_url,illegal_content_legal_ground,illegal_content_explanation,incompatible_content_ground,incompatible_content_explanation,incompatible_content_illegal,category,category_addition,category_specification,category_specification_other,content_type,content_type_other,content_language,content_date,territorial_scope,application_date,decision_facts,source_type,source_identity,automated_detection,automated_decision,platform_name,platform_uid,created_at
str,str,str,datetime[ms],str,str,datetime[ms],str,datetime[ms],str,datetime[ms],str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,datetime[ms],str,datetime[ms],str,str,str,str,str,str,str,datetime[ms]
"""b32035bf-154a-45d8-a0a4-f74030…","""[""DECISION_VISIBILITY_OTHER""]""","""Video not eligible for recomme…",null,null,null,null,null,null,null,null,null,"""DECISION_GROUND_INCOMPATIBLE_C…",null,null,null,"""Alcohol, Tobacco, and Drugs""","""While adults make personal cho…",null,"""STATEMENT_CATEGORY_SCOPE_OF_PL…",null,null,null,"""[""CONTENT_TYPE_VIDEO""]""",null,null,2024-12-31 00:00:00,"""[""AT"",""BE"",""BG"",""CY"",""CZ"",""DE""…",2024-12-31 00:00:00,"""The decision was taken pursuan…","""SOURCE_VOLUNTARY""",null,"""Yes""","""AUTOMATED_DECISION_FULLY""","""TikTok""","""7454722994082978593""",2025-01-01 00:00:06
"""76db65b6-5017-4387-b21a-cb315f…","""[""DECISION_VISIBILITY_CONTENT_…",null,null,null,null,null,null,null,null,null,null,"""DECISION_GROUND_INCOMPATIBLE_C…",null,null,null,"""Harassment and Bullying""","""We welcome the respectful expr…",null,"""STATEMENT_CATEGORY_ILLEGAL_OR_…",null,null,null,"""[""CONTENT_TYPE_TEXT""]""",null,null,2024-12-31 00:00:00,"""[""AT"",""BE"",""BG"",""CY"",""CZ"",""DE""…",2024-12-31 00:00:00,"""The decision was taken pursuan…","""SOURCE_VOLUNTARY""",null,"""Yes""","""AUTOMATED_DECISION_FULLY""","""TikTok""","""7454721838254545697""",2025-01-01 00:00:06
"""df2eb0ae-d712-41a1-a70e-f63a71…","""[""DECISION_VISIBILITY_OTHER""]""","""Video not eligible for recomme…",null,null,null,null,null,null,null,null,null,"""DECISION_GROUND_INCOMPATIBLE_C…",null,null,null,"""Alcohol, Tobacco, and Drugs""","""While adults make personal cho…",null,"""STATEMENT_CATEGORY_SCOPE_OF_PL…",null,null,null,"""[""CONTENT_TYPE_VIDEO""]""",null,null,2024-12-31 00:00:00,"""[""AT"",""BE"",""BG"",""CY"",""CZ"",""DE""…",2024-12-31 00:00:00,"""The decision was taken pursuan…","""SOURCE_VOLUNTARY""",null,"""Yes""","""AUTOMATED_DECISION_FULLY""","""TikTok""","""7454722994082962209""",2025-01-01 00:00:06
"""13cbae83-3e14-43df-afbb-64114b…","""[""DECISION_VISIBILITY_CONTENT_…",null,null,null,null,null,null,null,null,null,null,"""DECISION_GROUND_INCOMPATIBLE_C…",null,null,null,"""Hate Speech and Hateful Behavi…","""TikTok is enriched by the vari…",null,"""STATEMENT_CATEGORY_ILLEGAL_OR_…",null,null,null,"""[""CONTENT_TYPE_TEXT""]""",null,null,2024-12-31 00:00:00,"""[""AT"",""BE"",""BG"",""CY"",""CZ"",""DE""…",2024-12-31 00:00:00,"""The decision was taken pursuan…","""SOURCE_VOLUNTARY""",null,"""Yes""","""AUTOMATED_DECISION_FULLY""","""TikTok""","""7454722991587384097""",2025-01-01 00:00:06
"""524d908d-5f4a-46be-b403-f1a682…","""[""DECISION_VISIBILITY_OTHER""]""","""Video not eligible for recomme…",null,null,null,null,null,null,null,null,null,"""DECISION_GROUND_INCOMPATIBLE_C…",null,null,null,"""Community Guidelines""","""We maintain content eligibilit…",null,"""STATEMENT_CATEGORY_SCOPE_OF_PL…",null,null,null,"""[""CONTENT_TYPE_VIDEO""]""",null,null,2024-12-31 00:00:00,"""[""AT"",""BE"",""BG"",""CY"",""CZ"",""DE""…",2024-12-31 00:00:00,"""The decision was taken pursuan…","""SOURCE_VOLUNTARY""",null,"""Yes""","""AUTOMATED_DECISION_FULLY""","""TikTok""","""7454722994083027745""",2025-01-

### X (2025-01-01, first 5 rows)

In [7]:
pl.scan_parquet(str(X_SAMPLE_FILE)).head(5).collect()

uuid,decision_visibility,decision_visibility_other,end_date_visibility_restriction,decision_monetary,decision_monetary_other,end_date_monetary_restriction,decision_provision,end_date_service_restriction,decision_account,end_date_account_restriction,account_type,decision_ground,decision_ground_reference_url,illegal_content_legal_ground,illegal_content_explanation,incompatible_content_ground,incompatible_content_explanation,incompatible_content_illegal,category,category_addition,category_specification,category_specification_other,content_type,content_type_other,content_language,content_date,territorial_scope,application_date,decision_facts,source_type,source_identity,automated_detection,automated_decision,platform_name,platform_uid,created_at
str,str,str,datetime[ms],str,str,datetime[ms],str,datetime[ms],str,datetime[ms],str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,datetime[ms],str,datetime[ms],str,str,str,str,str,str,str,datetime[ms]
"""92d0aba5-2142-4593-88f0-5bc98e…","""[""DECISION_VISIBILITY_OTHER""]""","""Bounce""",null,null,null,null,"""DECISION_PROVISION_PARTIAL_SUS…",null,null,null,null,"""DECISION_GROUND_ILLEGAL_CONTEN…",null,"""AbusePolicyNonconsensualNudity""","""AbusePolicyNonconsensualNudity""",null,null,null,"""STATEMENT_CATEGORY_NON_CONSENS…",null,"""[""KEYWORD_NON_CONSENSUAL_IMAGE…",null,"""[""CONTENT_TYPE_SYNTHETIC_MEDIA…",null,null,2024-12-31 00:00:00,"""[""DE""]""",2024-12-31 00:00:00,"""Decision - action Bounce trigg…","""SOURCE_TYPE_OTHER_NOTIFICATION""",null,"""No""","""AUTOMATED_DECISION_NOT_AUTOMAT…","""X""","""AISAAAAAAAAAAAAAAAAAAAAAAAAAAA…",2025-01-01 00:47:03
"""4bdbc619-4b1a-41d6-aaaf-745504…","""[""DECISION_VISIBILITY_OTHER""]""","""Suspend""",null,null,null,null,"""DECISION_PROVISION_TOTAL_SUSPE…",null,"""DECISION_ACCOUNT_SUSPENDED""",null,null,"""DECISION_GROUND_ILLEGAL_CONTEN…",null,"""Cse""","""Cse""",null,null,null,"""STATEMENT_CATEGORY_PROTECTION_…",null,"""[""KEYWORD_CHILD_SEXUAL_ABUSE_M…",null,"""[""CONTENT_TYPE_SYNTHETIC_MEDIA…",null,null,2024-12-31 00:00:00,"""[""FR""]""",2024-12-31 00:00:00,"""Decision - action Suspend trig…","""SOURCE_TYPE_OTHER_NOTIFICATION""",null,"""No""","""AUTOMATED_DECISION_NOT_AUTOMAT…","""X""","""AISAAAAAAAAAAAAAAAAAAAAAAAAAAA…",2025-01-01 00:47:04
"""ae8a5539-c9a9-43ba-bda4-084436…","""[""DECISION_VISIBILITY_OTHER""]""","""Nsfw""",null,null,null,null,null,null,null,null,null,"""DECISION_GROUND_ILLEGAL_CONTEN…",null,"""NsfwViolenceGore""","""NsfwViolenceGore""",null,null,null,"""STATEMENT_CATEGORY_VIOLENCE""",null,"""[""KEYWORD_OTHER""]""",null,"""[""CONTENT_TYPE_SYNTHETIC_MEDIA…",null,null,2024-12-31 00:00:00,"""[""FR""]""",2024-12-31 00:00:00,"""Decision - action Nsfw trigger…","""SOURCE_TYPE_OTHER_NOTIFICATION""",null,"""No""","""AUTOMATED_DECISION_NOT_AUTOMAT…","""X""","""AISAAAAAAAAAAAAAAAAAAAAAAAAAAA…",2025-01-01 00:47:04
"""6af85efd-599a-4568-b2aa-61ded0…","""[""DECISION_VISIBILITY_OTHER""]""","""Bounce""",null,null,null,null,"""DECISION_PROVISION_PARTIAL_SUS…",null,null,null,null,"""DECISION_GROUND_ILLEGAL_CONTEN…",null,"""AbuseOneOff""","""AbuseOneOff""",null,null,null,"""STATEMENT_CATEGORY_SCOPE_OF_PL…",null,"""[""KEYWORD_OTHER""]""","""Defaulted to category ScopeOfP…","""[""CONTENT_TYPE_SYNTHETIC_MEDIA…",null,null,2024-12-31 00:00:00,"""[""HU""]""",2024-12-31 00:00:00,"""Decision - action Bounce trigg…","""SOURCE_TYPE_OTHER_NOTIFICATION""",null,"""No""","""AUTOMATED_DECISION_NOT_AUTOMAT…","""X""","""AISAAAAAAAAAAAAAAAAAAAAAAAAAAA…",2025-01-01 00:47:20
"""0a6455a2-24d1-4a8a-8414-dc5cd1…","""[""DECISION_VISIBILITY_OTHER""]""","""Nsfw""",null,null,null,null,null,null,null,null,null,"""DECISION_GROUND_ILLEGAL_CONTEN…",null,"""NsfwAdultContent""","""NsfwAdultContent""",null,null,null,"""STATEMENT_CATEGORY_PORNOGRAPHY…",null,"""[""KEYWORD_ADULT_SEXUAL_MATERIA…",null,"""[""CONTENT_TYPE_SYNTHETIC_MEDIA…",null,null,2024-12-31 00:00:00,"""[""FR""]""",2024-12-31 00:00:00,"""Decision - action Nsfw trigger…","""SOURCE_TYPE_OTHER_NOTIFICATION""",null,

## 5. Global Territorial Scope

The raw data covers all EEA countries -- not just Germany. This cell shows the variety of territorial scope values present before any filtering.

In [8]:
# Sample from one day to show territorial scope diversity (global, pre-filter)
scope_sample = (
    pl.scan_parquet(str(TIKTOK_SAMPLE_FILE))
    .select("territorial_scope")
    .unique()
    .head(20)
    .collect()
)
scope_sample

territorial_scope
str
"""[""ES"",""IE"",""IT"",""NL""]"""
"""[""ES"",""HR"",""IT""]"""
"""[""DE"",""NL""]"""
"""[""AT"",""BE"",""DK"",""ES"",""FI"",""HR""…"
"""[""CY"",""DE"",""FI"",""FR"",""HR"",""HU""…"
…
"""[""ES"",""FR"",""NL"",""RO""]"""
"""[""BE"",""ES"",""IT"",""PT""]"""
"""[""DE"",""ES"",""IE""]"""


## 6. Key Observations from the Raw Data

- **Global scope before filtering:** Raw `territorial_scope` contains records for all EEA countries. The DE filter (Notebook 02) reduced TikTok by 0.4% and X by 72.6%, revealing a fundamental difference in enforcement strategy between the two platforms.
- **Raw `category` labels are not harmonized:** The same violation type appears under different labels in v1 and v2. This is addressed in Notebook 03.